In [5]:
from manim import *
import numpy as np

# ---------------------------------------------------------

# CONFIGURACIÓN VERTICAL TIKTOK / SHORTS

# ---------------------------------------------------------

config.pixel_width = 1080
config.pixel_height = 1920
config.frame_rate = 60

# ---------------------------------------------------------

# ESCENA

# ---------------------------------------------------------

class CliffordTorusNeon(ThreeDScene):

# -----------------------------------------------------
# PROYECCIÓN ESTEREOGRÁFICA
# -----------------------------------------------------

    def stereographic(self, x, y, z, w):

        eps = 1e-4
        denom = np.maximum(1 - w, eps)

        return np.array([
            x / denom,
            y / denom,
            z / denom
        ])

    # -----------------------------------------------------
    # PUNTO DEL TORO DE CLIFFORD EN R4
    # -----------------------------------------------------

    def torus_point(self, u, v, theta):

        x = np.cos(u)
        y = np.sin(u)

        z = np.cos(v)
        w = np.sin(v)

        # Rotación 4D
        z2 = z*np.cos(theta) - w*np.sin(theta)
        w2 = z*np.sin(theta) + w*np.cos(theta)

        return self.stereographic(
            x,
            y,
            z2,
            w2
        )

    # -----------------------------------------------------
    # COLOR SEGÚN COORDENADA 4D
    # -----------------------------------------------------

    def color_from_parameter(self, t):

        palette = [
            BLUE_C,
            TEAL,
            GREEN,
            YELLOW,
            ORANGE,
            PINK,
            PURPLE
        ]

        n = len(palette) - 1

        idx = min(int(t*n), n-1)

        alpha = (t*n) - idx

        return interpolate_color(
            palette[idx],
            palette[idx+1],
            alpha
        )

    # -----------------------------------------------------
    # CURVA CON GLOW
    # -----------------------------------------------------

    def neon_curve(self, points, color):

        group = VGroup()

        glow_widths = [18, 12, 8, 4]

        glow_opacity = [0.05, 0.10, 0.20, 1]

        for w, op in zip(glow_widths, glow_opacity):

            c = VMobject()

            c.set_points_smoothly(points)

            c.set_stroke(
                color=color,
                width=w,
                opacity=op
            )

            group.add(c)

        return group

    # -----------------------------------------------------
    # RETÍCULA NEÓN
    # -----------------------------------------------------

    def create_grid(self):

        grid = VGroup()

        rng = np.arange(-6, 7, 1)

        for x in rng:

            line = Line(
                [x, -6, 0],
                [x, 6, 0]
            )

            line.set_stroke(
                BLUE_E,
                width=1,
                opacity=0.3
            )

            grid.add(line)

        for y in rng:

            line = Line(
                [-6, y, 0],
                [6, y, 0]
            )

            line.set_stroke(
                BLUE_E,
                width=1,
                opacity=0.3
            )

            grid.add(line)

        return grid

    # -----------------------------------------------------
    # TORO COMPLETO
    # -----------------------------------------------------

    def create_torus(self, theta):

        group = VGroup()

        samples = 140

        us = np.linspace(0, TAU, samples)
        vs = np.linspace(0, TAU, samples)

        density = 22

        # curvas U

        for i, v in enumerate(np.linspace(0, TAU, density)):

            pts = [
                self.torus_point(u, v, theta)
                for u in us
            ]

            color = self.color_from_parameter(
                i/(density-1)
            )

            group.add(
                self.neon_curve(
                    pts,
                    color
                )
            )

        # curvas V

        for i, u in enumerate(np.linspace(0, TAU, density)):

            pts = [
                self.torus_point(u, v, theta)
                for v in vs
            ]

            color = self.color_from_parameter(
                i/(density-1)
            )

            group.add(
                self.neon_curve(
                    pts,
                    color
                )
            )

        return group

    # -----------------------------------------------------
    # DOS CÍRCULOS INICIALES
    # -----------------------------------------------------

    def create_generator_circles(self):

        c1 = Circle(radius=1.8)

        c1.rotate(
            PI/2,
            axis=RIGHT
        )

        c1.set_color(TEAL)

        c2 = Circle(radius=1.8)

        c2.rotate(
            PI/2,
            axis=UP
        )

        c2.set_color(PURPLE)

        return VGroup(c1, c2)

    # -----------------------------------------------------
    # ESCENA PRINCIPAL
    # -----------------------------------------------------

    def construct(self):

        self.camera.background_color = BLACK

        # -----------------------------------------
        # CÁMARA
        # -----------------------------------------

        self.set_camera_orientation(
            phi=70 * DEGREES,
            theta=-45 * DEGREES,
            zoom=1.1
        )

        # -----------------------------------------
        # TÍTULOS
        # -----------------------------------------

        title = MathTex(
            r"T^2=\{(\cos u,\sin u,\cos v,\sin v)\subset\mathbb{R}^4\}"
        ).scale(0.55)

        projection = MathTex(
            r"\pi(x,y,z,w)=\left(\frac{x}{1-w},\frac{y}{1-w},\frac{z}{1-w}\right)"
        ).scale(0.52)

        projection.set_color(YELLOW)

        formulas = VGroup(
            title,
            projection
        ).arrange(
            DOWN,
            buff=0.25
        )

        formulas.to_edge(UP)

        panel = RoundedRectangle(
            width=8.5,
            height=1.6,
            corner_radius=0.15
        )

        panel.set_fill(
            BLACK,
            opacity=0.7
        )

        panel.set_stroke(
            BLUE_D,
            width=2
        )

        panel.move_to(formulas)

        self.add_fixed_in_frame_mobjects(
            panel,
            formulas
        )

        # -----------------------------------------
        # GRID
        # -----------------------------------------

        grid = self.create_grid()

        axes = ThreeDAxes(
            x_length=8,
            y_length=8,
            z_length=8
        )

        axes.set_stroke(
            opacity=0.5
        )

        self.add(grid)
        self.add(axes)

        # -----------------------------------------
        # TRANSICIÓN CÍRCULOS
        # -----------------------------------------

        circles = self.create_generator_circles()

        self.play(
            Create(circles),
            run_time=2
        )

        self.wait(1)

        theta_tracker = ValueTracker(0)

        torus = always_redraw(
            lambda:
            self.create_torus(
                theta_tracker.get_value()
            )
        )

        self.play(
            FadeOut(circles),
            FadeIn(torus),
            run_time=3
        )

        # -----------------------------------------
        # ANIMACIÓN 4D
        # -----------------------------------------

        self.begin_ambient_camera_rotation(
            rate=0.05
        )

        self.play(
            theta_tracker.animate.set_value(
                2 * PI
            ),
            run_time=25,
            rate_func=linear
        )

        self.wait(2)
